> LangChain Level 1：从基础检索到可引用、可拒答的工程进度查询

这个 notebook 只处理仓库现有的 **8 份输变电工程进度 TXT**，实现一条确定性的基础检索流水线：

**查询规范化 → 工程 metadata route → 中文词面/BM25 召回 → Dense 补充召回 → Weighted RRF → 完整任务记录定位 → Evidence Gate → 引用回答或拒答**

> 不用 LangGraph，不用 Agent，也不在查询时调用 LLM：

- Retriever 只负责召回候选 chunk，不能直接宣布日期正确。
- 工程名、任务名、任务标识号和完整任务段是强约束；语义相似度只是补充信号。
- 只有 Evidence Gate 同时确认唯一记录、字段完整、引用完整、证据 chunk 确实被召回时，系统才输出事实。
- 无法消歧、知识库没有记录、字段缺失或证据未召回时，系统明确拒答。

LangChain 官方文档把 VectorStore 定义为统一的增删查接口，并说明 k、filter 和相似度度量由具体存储实现决定：
https://docs.langchain.com/oss/python/integrations/vectorstores

Retriever 是 Runnable，可以 invoke/batch；VectorStore 本身不是 Runnable：
https://docs.langchain.com/oss/python/langchain/knowledge-base

In [ ]:
from __future__ import annotations

import json
import os
import sys
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from collections.abc import Callable, Mapping, Sequence
from typing import Any


os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")


def find_repo_root(start: Path | None = None) -> Path:
    """从仓库根目录或 ZZworkbench 启动时都能定位真实资源。"""

    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        corpus = candidate / "knowledge" / "project_progress" / "texts"
        if corpus.is_dir() and (candidate / "ZZworkbench").is_dir():
            return candidate
    raise FileNotFoundError("Cannot locate the pipelines_rag repository root")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from rank_bm25 import BM25Okapi

from ZZworkbench.project_progress_reliable import (
    ProjectProgressKnowledgeBase,
    QueryIntent,
    ReliableQueryResult,
    evaluate_reliable_lookup,
    normalize_lookup_text,
)
from ZZworkbench.rag_langchain.text_retrieval import (
    DEFAULT_CORPUS_ROOT,
    ChunkConfig,
    RetrievalHit,
    chinese_lexical_tokens,
    load_eval_cases,
    load_txt_documents,
    split_documents,
)

print("repository located:", REPO_ROOT.name == "pipelines_rag")

## 2. 加载固定语料，并审计完整任务记录

这批数据不是开放域百科，而是小型、半结构化的工程计划：

- 工程名称决定文档范围；
- 任务名可能在不同工程中重复；
- 日期、工期、父子任务和任务标识号必须来自同一条完整记录；
- chunk 只是检索单位，ScheduleRecord 才是事实单位。

因此先建立 documents → chunks → records 的可追溯关系。下面的审计若不为 0，后续任何 retriever 调参都不能修复数据切分问题。

In [ ]:
txt_dir = REPO_ROOT / "knowledge" / "project_progress" / "texts" / "v4"
txt_files = sorted(txt_dir.glob("*.txt"))

chunk_config = ChunkConfig()
documents = load_txt_documents(DEFAULT_CORPUS_ROOT, version="v4")
chunks = split_documents(documents, chunk_config)
knowledge_base = ProjectProgressKnowledgeBase(documents, chunks)

chunk_ids = {str(chunk.id or chunk.metadata.get("chunk_id")) for chunk in chunks}
unmapped_records = [
    record.record_id
    for record in knowledge_base.records
    if record.chunk_id not in chunk_ids
]

print(
    {
        "txt_files": len(txt_files),
        "documents": len(documents),
        "chunks": len(chunks),
        "schedule_records": len(knowledge_base.records),
        "records_without_chunk": len(unmapped_records),
        "deterministic_first_file": txt_files[0].name,
    }
)
assert not unmapped_records

## 3. 查询规范化与工程 metadata route

**规范化不是让 LLM 改写问题。** 当前实现只做确定性的 Unicode NFKC、大小写与空白/标点归一，并从仓库已维护的工程别名表中解析工程范围。

route 的价值在于先缩小合法证据空间。例如“江湾总体计划”和“江湾施工进度”是两份不同文档；若先在全部 chunk 上做语义 Top-K，另一个江湾文档很容易占据候选。

没有显式工程名时不猜工程，也不强行过滤；后面的 Evidence Gate 会对同名任务返回 ambiguous。

In [ ]:
route_examples = [
    "珠海110kV黄金输变电工程的主体结构封顶计划什么时候完成？",
    "施工准备什么时候完成？",
    "南溪（旅游）输变电工程的地基基础施工计划起止时间是什么？",
]

for query in route_examples:
    intent = knowledge_base.parse_query(query)
    print(
        json.dumps(
            {
                "original": query,
                "normalized": intent.normalized_query,
                "task_hint": intent.task_hint,
                "requested_fields": intent.requested_fields,
                "project_sources": [Path(source).name for source in intent.project_candidates],
                "query_type": intent.query_type,
            },
            ensure_ascii=False,
        )
    )

## 4. 中文 BM25：分词为什么会决定召回结果

BM25 的输入不是原始字符串，而是 token 序列。英文按空格切分通常尚可；中文句子没有天然空格，直接调用 split() 往往会把整句话当成一个 token，查询 token 与文档 token 很难完全相同。

本知识库采用一个可解释、零额外依赖的词面方案：

- 中文连续片段生成重叠 2-gram 和 3-gram；
- 英文、日期、任务编号和 110kV 等标签保留为归一化 token；
- 文档 title 与 chunk 正文一起进入 BM25，增强工程名和任务名。

这不是通用中文分词器，但对当前“工程名 + 任务名 + 日期”的小语料很合适。若未来加入同义词丰富的自然语言文档，应通过评测比较 jieba、pkuseg 或领域词典，而不是凭感觉替换。

In [ ]:
token_demo = "南溪（旅游）输变电工程的地基基础施工计划，110kV，2025年8月2日"
print("str.split():", token_demo.casefold().split())
print("project tokenizer:", chinese_lexical_tokens(token_demo)[:36])

In [ ]:
class RoutedChineseBM25:
    """当前 63 个 chunk 的小型内存 BM25；底层算法来自 rank_bm25。"""

    def __init__(self, documents: Sequence[Document]) -> None:
        self.documents = list(documents)
        corpus_tokens = [
            chinese_lexical_tokens(
                f"{document.metadata.get('title', '')}\n{document.page_content}"
            )
            for document in self.documents
        ]
        self.index = BM25Okapi(corpus_tokens)

    def search(
        self,
        query: str,
        *,
        k: int = 20,
        allowed_sources: set[str] | None = None,
    ) -> list[RetrievalHit]:
        """先按 source_name route，再在合法 chunk 中按 BM25 分数排序。"""

        tokens = chinese_lexical_tokens(query)
        if not tokens:
            return []
        raw_scores = self.index.get_scores(tokens)
        eligible = [
            (index, float(score))
            for index, score in enumerate(raw_scores)
            if (
                allowed_sources is None
                or str(self.documents[index].metadata.get("source_name")) in allowed_sources
            )
            and float(score) > 0
        ]
        ranked = sorted(
            eligible,
            key=lambda item: (
                -item[1],
                str(
                    self.documents[item[0]].id
                    or self.documents[item[0]].metadata.get("chunk_id")
                ),
            ),
        )[:k]

        hits: list[RetrievalHit] = []
        for rank, (index, score) in enumerate(ranked, start=1):
            document = self.documents[index]
            annotated = Document(
                id=document.id,
                page_content=document.page_content,
                metadata={
                    **document.metadata,
                    "lexical_rank": rank,
                    "lexical_score": score,
                },
            )
            hits.append(
                RetrievalHit(
                    document=annotated,
                    rank=rank,
                    score=score,
                    distance=None,
                )
            )
        return hits


bm25_index = RoutedChineseBM25(chunks)
print("BM25 indexed chunks:", len(bm25_index.documents))

## 5. Dense：InMemoryVectorStore 与本地中文 embedding

这里按要求使用 **langchain_core.vectorstores.InMemoryVectorStore** 和本地
**iic--nlp_gte_sentence-embedding_chinese-base**。

InMemoryVectorStore 适合当前 63 个 chunk 的教学与回归测试：

- 全部向量只放在当前 Python 进程内，重启即消失；
- 当前实现做精确线性扫描，不是 ANN 索引；
- 当前实现用 cosine similarity，分数越大越相似；
- 换成 Chroma、Qdrant 或其他 VectorStore 后，score 可能表示距离，方向和尺度必须重新确认。

因此 search_type 不负责选择 Cosine / Inner Product / L2；距离度量由具体 VectorStore/索引配置决定。

In [ ]:
embed_model_name = "iic--nlp_gte_sentence-embedding_chinese-base"
embed_path = Path("/mnt/e/local_models/embedding") / embed_model_name
if not embed_path.is_dir():
    raise FileNotFoundError(
        "Local embedding model is missing; expected model directory under /mnt/e/local_models/embedding"
    )

embeddings = HuggingFaceEmbeddings(
    model=str(embed_path),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
    query_encode_kwargs={"normalize_embeddings": True},
    show_progress=False,
)

vector_store = InMemoryVectorStore(embedding=embeddings)
stable_ids = [str(chunk.id or chunk.metadata["chunk_id"]) for chunk in chunks]
indexed_ids = vector_store.add_documents(documents=chunks, ids=stable_ids)

print(
    {
        "embedding_model": embed_path.name,
        "normalized_vectors": True,
        "indexed_chunks": len(indexed_ids),
        "store": type(vector_store).__name__,
    }
)

## 6. VectorStore 的基础检索接口

下面分别演示：

- similarity_search：只返回 Document；
- similarity_search_with_score：返回 (Document, score)，是调试、融合和阈值校准更有用的接口；
- similarity_search_by_vector：已有 query embedding 时避免重复编码；
- get_by_ids：验证稳定 ID 能回读。

对本 notebook 的 InMemoryVectorStore，with_score 的 score 是 cosine similarity，越大越相关。不要把这条结论机械迁移到其他 VectorStore。

In [ ]:
api_query = "珠海黄金输变电工程的主体结构封顶计划什么时候完成？"
api_intent = knowledge_base.parse_query(api_query)
api_sources = set(api_intent.project_candidates)
api_filter = lambda document: (
    not api_sources
    or str(document.metadata.get("source_name")) in api_sources
)

plain_docs = vector_store.similarity_search(
    api_intent.normalized_query,
    k=3,
    filter=api_filter,
)
scored_docs = vector_store.similarity_search_with_score(
    api_intent.normalized_query,
    k=3,
    filter=api_filter,
)
query_vector = embeddings.embed_query(api_intent.normalized_query)
by_vector_docs = vector_store.similarity_search_by_vector(
    query_vector,
    k=2,
    filter=api_filter,
)
fetched_docs = vector_store.get_by_ids(stable_ids[:2])

print("similarity_search chunk_ids:", [doc.metadata["chunk_id"] for doc in plain_docs])
print(
    "similarity_search_with_score:",
    [
        {
            "chunk_id": doc.metadata["chunk_id"],
            "cosine_similarity": round(float(score), 4),
        }
        for doc, score in scored_docs
    ],
)
print("similarity_search_by_vector count:", len(by_vector_docs))
print("get_by_ids count:", len(fetched_docs))

## 7. as_retriever：similarity、MMR 与 score threshold

as_retriever 把 VectorStore 包装成 LangChain Runnable，因此统一使用 invoke、batch、ainvoke 等接口。

**similarity**

- k：最终返回数量；
- filter：当前 InMemoryVectorStore 接受 callable，先做工程 metadata route；
- 适合精确事实问答基线。

**mmr（Maximal Marginal Relevance）**

- fetch_k：先做向量搜索取得的候选池，必须明显大于 k 才有意义；
- lambda_mult：1 更偏相关，0 更偏多样；这里取 0.65；
- MMR 不是新的 embedding 或 ANN 索引，只是在 Dense 候选上重新选择，适合多方面概括，不保证单条日期问答更准。

**similarity_score_threshold**

- score_threshold 只属于此 search_type；
- 它避免“Top-K 无论有没有答案都强制返回 K 条”的问题；
- 阈值必须根据当前 embedding、VectorStore 和标注集校准，不能从别的模型复制。

还要尊重具体实现：当前项目安装版的 InMemoryVectorStore 没有实现 relevance-score 映射，
因而 threshold retriever 在 invoke 时会抛 NotImplementedError。下面对 similarity 与 MMR
使用 as_retriever；阈值演示改用 similarity_search_with_score 返回的原始 cosine
similarity 手动过滤。换 VectorStore 后要重新确认 score 的方向和尺度。

In [ ]:
similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": api_filter,
    },
)
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 16,
        "lambda_mult": 0.65,
        "filter": api_filter,
    },
)
retriever_demo = {
    "similarity": similarity_retriever.invoke(api_intent.normalized_query),
    "mmr": mmr_retriever.invoke(api_intent.normalized_query),
}
manual_threshold = 0.55
manual_threshold_docs = [
    document
    for document, score in vector_store.similarity_search_with_score(
        api_intent.normalized_query,
        k=8,
        filter=api_filter,
    )
    if float(score) >= manual_threshold
]
retriever_demo["manual_cosine_threshold"] = manual_threshold_docs
print(
    {
        name: [doc.metadata["chunk_id"] for doc in docs]
        for name, docs in retriever_demo.items()
    }
)

## 8. Dense 补充召回

Dense 分支也先使用同一个工程 route。它的职责是补充词面表达不完全一致的候选，不覆盖 BM25 的强实体信号。

这里直接调用 similarity_search_with_score，而不是只调用 retriever.invoke，因为融合阶段需要保留原始分数和分支排名。Weighted RRF 实际只使用排名，原始 Dense/BM25 score 留作诊断。

In [ ]:
def dense_search(
    query: str,
    *,
    allowed_sources: set[str] | None,
    k: int = 20,
) -> list[RetrievalHit]:
    source_filter: Callable[[Document], bool] = lambda document: (
        allowed_sources is None
        or str(document.metadata.get("source_name")) in allowed_sources
    )
    pairs = vector_store.similarity_search_with_score(
        query,
        k=k,
        filter=source_filter,
    )
    hits: list[RetrievalHit] = []
    for rank, (document, score) in enumerate(pairs, start=1):
        annotated = Document(
            id=document.id,
            page_content=document.page_content,
            metadata={
                **document.metadata,
                "dense_rank": rank,
                "dense_score": float(score),
            },
        )
        hits.append(
            RetrievalHit(
                document=annotated,
                rank=rank,
                score=float(score),
                distance=None,
            )
        )
    return hits

## 9. Weighted RRF：融合排名，不混加不可比分数

BM25 score 和 cosine similarity 的量纲、范围不同，直接做
0.65 × BM25 score + 0.35 × cosine score 没有稳定含义。

Reciprocal Rank Fusion 只依赖各分支排名：

**RRF(d) = Σ weight(branch) / (rrf_k + rank(branch, d))**

当前语料高度依赖工程名、任务名、编号和日期词面，因此先给 BM25 0.65、Dense 0.35。这个权重是面向当前知识库的可评测起点，不是通用最佳值。rrf_k=60 会让相邻名次差异较平滑；它不是最终返回 k。

In [ ]:
def _chunk_key(document: Document) -> str:
    return str(document.id or document.metadata.get("chunk_id"))


def weighted_rrf(
    branches: Mapping[str, Sequence[RetrievalHit]],
    *,
    weights: Mapping[str, float],
    k: int = 8,
    rrf_k: int = 60,
) -> list[RetrievalHit]:
    if rrf_k < 1:
        raise ValueError("rrf_k must be positive")

    fused_scores: dict[str, float] = defaultdict(float)
    documents_by_id: dict[str, Document] = {}
    signals: dict[str, dict[str, Any]] = defaultdict(dict)

    for branch_name, hits in branches.items():
        weight = float(weights[branch_name])
        for rank, hit in enumerate(hits, start=1):
            chunk_id = _chunk_key(hit.document)
            documents_by_id.setdefault(chunk_id, hit.document)
            fused_scores[chunk_id] += weight / (rrf_k + rank)
            signals[chunk_id][f"{branch_name}_rank"] = rank
            signals[chunk_id][f"{branch_name}_score"] = float(hit.score)

    ordered_ids = sorted(
        fused_scores,
        key=lambda chunk_id: (-fused_scores[chunk_id], chunk_id),
    )[:k]

    fused_hits: list[RetrievalHit] = []
    for rank, chunk_id in enumerate(ordered_ids, start=1):
        document = documents_by_id[chunk_id]
        score = fused_scores[chunk_id]
        annotated = Document(
            id=document.id,
            page_content=document.page_content,
            metadata={
                **document.metadata,
                **signals[chunk_id],
                "rrf_rank": rank,
                "rrf_score": score,
            },
        )
        fused_hits.append(
            RetrievalHit(
                document=annotated,
                rank=rank,
                score=score,
                distance=None,
            )
        )
    return fused_hits

## 10. 串起 Level 1 检索 pipeline

每次查询只走一条有界、可观察的路径：

1. parse_query 规范化问题，解析工程、任务和请求字段；
2. project_candidates 转为允许的 source_name 集合；
3. BM25 和 Dense 各取 20 个 route 后候选；
4. Weighted RRF 合并为最多 8 个候选；
5. 完整任务记录定位与 Evidence Gate 在下一节执行。

route 后候选少于 fetch_k 是正常现象；fetch_k 是上限，不是必须凑满的数量。

In [ ]:
@dataclass(frozen=True)
class Level1Retrieval:
    intent: QueryIntent
    lexical_hits: tuple[RetrievalHit, ...]
    dense_hits: tuple[RetrievalHit, ...]
    fused_hits: tuple[RetrievalHit, ...]

    def trace(self) -> dict[str, Any]:
        return {
            "normalized_query": self.intent.normalized_query,
            "task_hint": self.intent.task_hint,
            "project_sources": list(self.intent.project_candidates),
            "lexical_chunk_ids": [_chunk_key(hit.document) for hit in self.lexical_hits],
            "dense_chunk_ids": [_chunk_key(hit.document) for hit in self.dense_hits],
            "fused": [
                {
                    "rank": hit.rank,
                    "chunk_id": _chunk_key(hit.document),
                    "source_name": hit.document.metadata.get("source_name"),
                    "rrf_score": round(hit.score, 6),
                    "lexical_rank": hit.document.metadata.get("lexical_rank"),
                    "dense_rank": hit.document.metadata.get("dense_rank"),
                }
                for hit in self.fused_hits
            ],
        }


def retrieve_level1(
    query: str,
    *,
    lexical_fetch_k: int = 20,
    dense_fetch_k: int = 20,
    final_k: int = 8,
    lexical_weight: float = 0.65,
    dense_weight: float = 0.35,
    rrf_k: int = 60,
) -> Level1Retrieval:
    intent = knowledge_base.parse_query(query)
    allowed_sources = (
        set(intent.project_candidates)
        if intent.project_candidates
        else None
    )
    lexical_hits = bm25_index.search(
        intent.normalized_query,
        k=lexical_fetch_k,
        allowed_sources=allowed_sources,
    )
    dense_hits = dense_search(
        intent.normalized_query,
        allowed_sources=allowed_sources,
        k=dense_fetch_k,
    )
    fused_hits = weighted_rrf(
        {
            "lexical": lexical_hits,
            "dense": dense_hits,
        },
        weights={
            "lexical": lexical_weight,
            "dense": dense_weight,
        },
        k=final_k,
        rrf_k=rrf_k,
    )
    return Level1Retrieval(
        intent=intent,
        lexical_hits=tuple(lexical_hits),
        dense_hits=tuple(dense_hits),
        fused_hits=tuple(fused_hits),
    )


exact_retrieval = retrieve_level1(api_query)
print(json.dumps(exact_retrieval.trace(), ensure_ascii=False, indent=2))

## 11. 完整任务记录定位与 Evidence Gate

Evidence Gate 复用项目的 ProjectProgressKnowledgeBase，按以下顺序判断：

- 是否识别到具体任务；
- 工程范围内是否存在任务记录；
- 是否跨工程或同文档多记录冲突；
- 请求字段是否齐全；
- source_name、task_id、chunk_id 引用是否完整；
- 最终唯一记录所在 chunk 是否真的出现在本次 fused candidates 中。

注意最后一条：结构化解析器即使能在全量 documents 中找到记录，若 retriever 本次没有召回该证据 chunk，也不能绕过检索链路直接回答。

In [ ]:
def run_level1(query: str, **retrieval_kwargs: Any) -> tuple[Level1Retrieval, ReliableQueryResult]:
    retrieval = retrieve_level1(query, **retrieval_kwargs)
    result = knowledge_base.lookup_intent(
        retrieval.intent,
        retriever=lambda _normalized_query: retrieval.fused_hits,
    )
    return retrieval, result


exact_retrieval, exact_result = run_level1(api_query)
print(
    json.dumps(
        {
            "status": exact_result.status,
            "answer": exact_result.answer,
            "retrieval_calls": exact_result.retrieval_calls,
            "evidence_gate": exact_result.diagnostics,
        },
        ensure_ascii=False,
        indent=2,
    )
)

## 12. 引用回答或拒答

下面同时覆盖三种行为：

- exact：唯一记录 + 字段 + 引用 + retrieved chunk 全部通过；
- ambiguous：任务存在于多个工程，要求补充工程范围；
- not_found：工程已确定，但知识库中没有这个任务。

拒答不是异常，而是可靠 pipeline 的正常输出状态。

In [ ]:
behavior_queries = [
    "珠海110kV黄金输变电工程的主体结构封顶计划什么时候完成？",
    "施工准备什么时候完成？",
    "南溪输变电工程的锅炉点火计划什么时候完成？",
    "110千伏节点计划中父任务试桩什么时候完成？",
]

behavior_rows = []
for query in behavior_queries:
    retrieval, result = run_level1(query)
    behavior_rows.append(
        {
            "query": query,
            "status": result.status,
            "fused_candidates": len(retrieval.fused_hits),
            "answer_or_refusal": result.answer,
        }
    )
print(json.dumps(behavior_rows, ensure_ascii=False, indent=2))

## 13. 一个必要的反例：候选未覆盖时不能回答

为了证明 Evidence Gate 不是装饰，下面故意只把无关 chunk 交给它。结构化索引仍能定位黄金工程的“主体结构封顶”，但证据 chunk 不在候选中，状态必须变成 insufficient。

In [ ]:
target_source = exact_result.records[0].source_name
wrong_document = next(
    chunk
    for chunk in chunks
    if chunk.metadata.get("source_name") != target_source
)
wrong_hit = RetrievalHit(
    document=wrong_document,
    rank=1,
    score=1.0,
    distance=None,
)
gate_demo = knowledge_base.lookup_intent(
    exact_retrieval.intent,
    retriever=lambda _normalized_query: [wrong_hit],
)
print(
    {
        "status": gate_demo.status,
        "answer": gate_demo.answer,
        "record_in_retrieved_chunks": gate_demo.diagnostics.get(
            "record_in_retrieved_chunks"
        ),
    }
)
assert gate_demo.status == "insufficient"

## 14. 用真实标注集验证，而不是用单个漂亮示例

当前评测由两部分组成：

- retrieval_v4：8 条真实工程事实查询；
- reliability_v4：exact、ambiguous、not_found、父任务与全字段查询，共 5 条。

评测检查 status、source、字段和证据词，而不仅是“返回了若干相似 chunk”。缓存只避免同一问题重复做 embedding，不改变任何检索结果。

In [ ]:
eval_dir = REPO_ROOT / "knowledge" / "project_progress" / "evals"
retrieval_cases = load_eval_cases(eval_dir / "retrieval_v4.jsonl")
reliability_cases = load_eval_cases(eval_dir / "reliability_v4.jsonl")
all_cases = [*retrieval_cases, *reliability_cases]

evaluation_cache = {
    normalize_lookup_text(str(case["query"])): retrieve_level1(
        str(case["query"])
    ).fused_hits
    for case in all_cases
}


def cached_retriever(normalized_query: str) -> Sequence[RetrievalHit]:
    return evaluation_cache[normalize_lookup_text(normalized_query)]


reliable_report = evaluate_reliable_lookup(
    knowledge_base,
    all_cases,
    retriever=cached_retriever,
)

print(
    json.dumps(
        {
            key: value
            for key, value in reliable_report.items()
            if key != "rows"
        },
        ensure_ascii=False,
        indent=2,
    )
)
failed_rows = [
    row for row in reliable_report["rows"] if not row["passed"]
]
print("failed cases:", failed_rows)
assert reliable_report["pass_rate"] == 1.0

## 15. 分支召回对照

最终可靠性评测通过仍不代表每个分支都同样好。下面在 8 条正例上分别检查 BM25、Dense 与 RRF top-8 是否同时覆盖 expected_source 和 expected_terms。

这个对照用于回答“语义检索是否可靠”：

- Dense 可以扩大表达召回，但不具备事实约束；
- BM25 对任务名、编号和日期词面更稳定，但依赖 tokenizer；
- RRF 降低单一路线失误的影响；
- Evidence Gate 才负责把候选升级为可回答证据。

In [ ]:
def candidate_passes(
    hits: Sequence[RetrievalHit],
    case: Mapping[str, Any],
    *,
    k: int = 8,
) -> bool:
    selected = list(hits)[:k]
    expected_source = case.get("expected_source")
    source_ok = expected_source is None or any(
        hit.document.metadata.get("source_name") == expected_source
        for hit in selected
    )
    evidence = "\n".join(hit.document.page_content for hit in selected)
    terms_ok = all(
        str(term) in evidence
        for term in case.get("expected_terms", [])
    )
    return source_ok and terms_ok


branch_passes = {"bm25": 0, "dense": 0, "weighted_rrf": 0}
branch_rows = []
for case in retrieval_cases:
    retrieval = retrieve_level1(str(case["query"]))
    checks = {
        "bm25": candidate_passes(retrieval.lexical_hits, case),
        "dense": candidate_passes(retrieval.dense_hits, case),
        "weighted_rrf": candidate_passes(retrieval.fused_hits, case),
    }
    for name, passed in checks.items():
        branch_passes[name] += int(passed)
    branch_rows.append({"id": case["id"], **checks})

print(
    {
        name: f"{passed}/{len(retrieval_cases)}"
        for name, passed in branch_passes.items()
    }
)
print(json.dumps(branch_rows, ensure_ascii=False, indent=2))

## 16. 工程结论与下一步调参顺序

这条 Level 1 pipeline 的可靠性来自分层职责，而不是来自某一个“高级”检索算法：

1. **先修数据合同**：完整任务记录必须能回指 chunk，引用字段必须稳定。
2. **先 route 再检索**：显式工程别名优先于全库语义近邻。
3. **BM25 负责词面锚点**：工程名、任务名、编号、日期；中文 tokenizer 是算法输入的一部分。
4. **Dense 负责表达补充**：相似度高只代表语义接近，不代表日期属于目标任务。
5. **RRF 融合排名**：避免直接混加不可比的 BM25 与 cosine 原始分数。
6. **完整记录 + Evidence Gate 决定回答权**：候选未覆盖、歧义、缺字段、缺引用都拒答。

对当前 63 个 chunk，InMemoryVectorStore 足够用于教学和回归。未来语料增大时，可以把 Dense 存储替换为持久化 ANN VectorStore，但 route、BM25、RRF、完整记录定位、Evidence Gate 和评测合同应保持不变。

推荐的调参顺序是：先看每条失败样例 → 修 route/tokenizer/数据 → 调 branch fetch_k → 调 BM25/Dense 权重 → 最后才校准 score_threshold。不要用一个人工挑选的问题决定全局阈值。